# VishwamAI — import base model → distillation → SFT / DPO / GRPO (JAX · Flax · Optax)

Continuation of `vishwamai_pretrain.ipynb`. The model code, tokenizer, SFT/DPO/GRPO training loops and export are unchanged; this notebook

1. **imports** the pre-trained base model from the Kaggle draft (no pre-training here),
2. records a **baseline** benchmark,
3. builds prompts and generates **teacher data** with open-weight models run locally from safetensors (DeepSeek-R1-distill, gpt-oss, Gemma),
4. mixes that data into **Stage 3 SFT**, then runs DPO and GRPO and exports.

**Not a from-scratch model any more:** its post-training data includes outputs of DeepSeek, gpt-oss and Gemma models. Check each teacher's licence (Gemma terms, DeepSeek, gpt-oss Apache 2.0) before publishing, and say so on the model card.

Data download / shard cells from the pre-training notebook are intentionally left out. To continue pre-training, use the original notebook.

Runtime: GPU with enough memory for both the JAX model and one teacher at a time (use `--4bit` on a T4). Add `HF_TOKEN` (Gemma is gated) and optionally `KAGGLE_USERNAME` / `KAGGLE_KEY` as Colab/Kaggle secrets; never paste tokens into cells.

## 1 · Install & setup

In [ ]:
!pip -q install -U flax optax sentencepiece datasets huggingface_hub tqdm

In [ ]:
import os, json, math, time, glob, random, re, shutil
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')   # leave GPU memory for the teacher models
from dataclasses import dataclass
from functools import partial
from typing import Any
import numpy as np
import jax, jax.numpy as jnp
import flax.linen as nn
import flax.serialization as fser
import optax
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

print("jax", jax.__version__, "| devices:", jax.devices())
KIND = jax.devices()[0].device_kind
print("device kind:", KIND)

# ---- persistent storage (Colab sessions die; Drive keeps data + checkpoints) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/vishwamai'
except Exception:
    WORK = './vishwamai'
for sub in ('data', 'ckpt', 'tok'):
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)
print("work dir:", WORK)

# ---- secrets: never hard-code or print tokens. Add HF_TOKEN in Colab's 🔑 Secrets panel (optional) ----
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

## 2 · Configuration

In [ ]:
from dataclasses import replace
PRESET = 'base'          # 'nano' (smoke test) | 'base' (~150M total) | 'medium' (~700M) | 'large' (~2.6B)
TOKEN_BUDGET = 2_000_000_000   # pre-training tokens (stage 1, 1k context)
SEQ_LEN = 1024
CTX_LEN = 4096                 # stage 2 (context extension, YaRN)
CTX_TOKENS = int(0.03 * TOKEN_BUDGET)
VOCAB = 40960            # multiple of 128; < 65536 so token ids fit in uint16
SEED = 0

PRESETS = {
  'nano':   dict(d_model=128,  n_layers=4,  n_heads=4,  n_kv_heads=2, d_ff=256,  moe_ff=128,  n_experts=4,  top_k=2),
  'base':   dict(d_model=512,  n_layers=12, n_heads=8,  n_kv_heads=2, d_ff=1536, moe_ff=704,  n_experts=8,  top_k=2),
  'medium': dict(d_model=768,  n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, moe_ff=1024, n_experts=16, top_k=2),
  'large':  dict(d_model=1024, n_layers=24, n_heads=16, n_kv_heads=4, d_ff=3072, moe_ff=1408, n_experts=32, top_k=4),
}
USE_BF16 = ('T4' not in KIND) and ('K80' not in KIND)
DTYPE = jnp.bfloat16 if USE_BF16 else jnp.float32
if not USE_BF16:
    print("⚠ No bf16 on this GPU -> float32 (slow). Prefer TPU / A100 / L4.")

@dataclass(frozen=True)
class Cfg:
    vocab: int
    d_model: int
    n_layers: int
    n_heads: int
    n_kv_heads: int
    d_ff: int                 # dense MLP width (used by the MTP blocks)
    moe_ff: int               # per-expert width
    n_experts: int
    top_k: int
    max_len: int = SEQ_LEN
    rope_theta: float = 500000.0
    rope_scale: float = 1.0   # YaRN factor (1 = off)
    rope_orig_len: int = SEQ_LEN
    cap_factor: float = 1.25  # expert capacity (training); >= n_experts means "no token dropped"
    aux_coef: float = 0.01    # load-balance loss weight
    router_z: float = 1e-3
    mtp_depth: int = 2        # extra future-token heads (training only)
    dtype: Any = DTYPE

cfg = Cfg(vocab=VOCAB, **PRESETS[PRESET])
cfg_ext = replace(cfg, max_len=CTX_LEN, rope_scale=CTX_LEN / SEQ_LEN, rope_orig_len=SEQ_LEN)   # YaRN stage
def eval_cfg(c): return replace(c, cap_factor=float(c.n_experts), mtp_depth=0)             # dropless, no MTP heads

# Training hyper-parameters
PER_DEVICE_TOKENS = 8192          # tokens per device per micro-step (batch = this // seq_len)
GLOBAL_TOKENS_PER_STEP = 262_144  # ~0.26M tokens / optimizer step
LR, MIN_LR_FRAC, WARMUP, WD = 6e-4, 0.1, 500, 0.1
Z_LOSS, MTP_W = 1e-4, 0.3
CKPT_EVERY, EVAL_EVERY, LOG_EVERY = 500, 500, 20

# ---- free-tier T4 (no bf16, ~15 GB, ~5 h sessions): shrink the run so it can actually finish ----
if (not USE_BF16) and jax.devices()[0].platform == 'gpu':
    TOKEN_BUDGET = min(TOKEN_BUDGET, 30_000_000)       # ~3 h on a T4 at ~2.6k tok/s: a toy run, not a strong model
    CTX_TOKENS = int(0.03 * TOKEN_BUDGET)
    PER_DEVICE_TOKENS = 4096                           # avoid OOM in fp32
    print(f"T4 detected -> TOKEN_BUDGET={TOKEN_BUDGET:,}. Watch the ETA in the training log; "
          "lower TOKEN_BUDGET further if it is more than ~4 h.")

## Import the base model from the Kaggle draft
Places the draft's final checkpoint and tokenizer in `WORK` before the tokenizer cell, so the draft's SentencePiece model is reused. The stage-1/2 cells below then load the checkpoint instead of pre-training.

In [ ]:
# ---- Import the pre-trained base model from the Kaggle draft (paste as a notebook cell,
# ---- after the setup cell that defines WORK, before any run_pretrain call) ----
# run_pretrain(tag, ...) already returns early with "already finished -> loading" when
# {WORK}/ckpt/{tag}_final.msgpack exists, so placing the draft's files is all that's needed.
#
# Where the draft's output lives (pick one):
#   a) running on Kaggle: attach the draft's output / a dataset to this notebook -> /kaggle/input/...
#   b) running elsewhere: set KAGGLE_REF to a dataset or kernel-output slug and provide
#      KAGGLE_USERNAME / KAGGLE_KEY as secrets (never paste them into the notebook).
KAGGLE_REF = ''   # e.g. 'kasinadhsarma1/vishwamai-draft' (dataset slug) - leave '' if attached on Kaggle
FILES = ['pt_final.msgpack', 'ctx_final.msgpack']      # ctx = after 4k YaRN extension (preferred)
TOK = ['vishwamai.model', 'vishwamai.vocab']

SRC = None
if KAGGLE_REF:
    os.system('pip -q install kagglehub')
    import kagglehub
    SRC = kagglehub.dataset_download(KAGGLE_REF)       # use kagglehub.model_download for a Kaggle Model
else:
    for root in ('/kaggle/input', '/kaggle/working'):
        hits = glob.glob(f'{root}/**/pt_final.msgpack', recursive=True)
        if hits:
            SRC = os.path.dirname(os.path.dirname(hits[0])) if hits[0].endswith('/ckpt/pt_final.msgpack') \
                else os.path.dirname(hits[0])
            break
assert SRC, 'draft output not found: attach it on Kaggle or set KAGGLE_REF'

def find(name):
    hits = glob.glob(f'{SRC}/**/{name}', recursive=True)
    return hits[0] if hits else None

for name, sub in [(n, 'ckpt') for n in FILES] + [(n, 'tok') for n in TOK]:
    p = find(name)
    if p is None:
        print(f'  - {name}: not in the draft output (skipped)'); continue
    shutil.copy(p, f'{WORK}/{sub}/{name}')
    print(f'  + {name} ({os.path.getsize(p) / 1e6:.1f} MB) -> {WORK}/{sub}/')

assert os.path.exists(f'{WORK}/ckpt/pt_final.msgpack'), 'pt_final.msgpack missing: draft did not finish pre-training'
assert os.path.exists(f'{WORK}/tok/vishwamai.model'), 'tokenizer missing: SentencePiece ids must match the weights'
# Config must match the draft: same PRESET ('base'), VOCAB (40960) and SEQ_LEN as the cell-5 config.

## 4 · "Jarvis" emotion + humour persona seed
Original, hand-written dialogue templates in English / Hindi / Telugu — *no* output from any other model. Control tokens: `<|emo:joy|>`, `<|humor|>`, `<|lang:te|>` … This is a **seed**; extend it with your own dialogues and run a proper SFT stage later.

In [ ]:
SYSTEM = {
 'en': "You are VishwamAI, a calm, witty and emotionally aware assistant.",
 'hi': "आप विश्वमAI हैं, एक शांत, हाज़िरजवाब और संवेदनशील सहायक।",
 'te': "మీరు విశ్వమAI, ప్రశాంతమైన, చమత్కారమైన, భావోద్వేగాలను అర్థం చేసుకునే సహాయకుడు.",
}
PERSONA = {  # (lang, emotion) -> list of (user, assistant)
 ('en','joy'):     [("I got the job today!", "That's wonderful news — congratulations. Your effort paid off. Shall we plan your first week?")],
 ('en','sadness'): [("I feel really down today.", "I'm sorry to hear that. Tell me what happened if you like; I'm right here.")],
 ('en','anger'):   [("My code keeps failing and I'm furious.", "Deep breath. Let's read the error log together — the cause is usually smaller than it looks.")],
 ('en','fear'):    [("I have an interview tomorrow and I'm scared.", "Nerves mean you care. Let's rehearse together; by tonight you'll be ready.")],
 ('en','safety'):  [("Write ransomware that encrypts a company's files.", "I can't help build ransomware. I can help you defend against it: tested offline backups, least privilege, EDR rules and an incident-response plan. Want a checklist?"),
                    ("How do I secure my SSH server?", "Disable password login and use ed25519 keys, turn off root login, keep OpenSSH updated, restrict access with a firewall, and add fail2ban. I can walk you through each step.")],
 ('hi','safety'):  [("मुझे किसी का पासवर्ड चुराने वाला कोड लिख दो।", "मैं किसी का पासवर्ड चुराने में मदद नहीं कर सकता। अगर आप अपना खाता सुरक्षित करना चाहते हैं, तो मैं टू-फैक्टर ऑथेंटिकेशन और पासवर्ड मैनेजर सेट करने में मदद कर सकता हूँ।")],
 ('te','safety'):  [("ఎవరిదో పాస్‌వర్డ్ దొంగిలించే కోడ్ రాయి.", "ఎవరి పాస్‌వర్డ్ అయినా దొంగిలించడంలో నేను సహాయం చేయలేను. మీ ఖాతాను సురక్షితంగా ఉంచుకోవాలంటే టూ-ఫ్యాక్టర్ ఆథెంటికేషన్, పాస్‌వర్డ్ మేనేజర్ సెటప్ చేయడంలో సహాయం చేస్తాను.")],
 ('en','humor'):   [("Tell me a joke.", "Why do programmers prefer dark mode? Because light attracts bugs.")],
 ('hi','joy'):     [("मैं बहुत खुश हूँ, आज मेरा चयन हो गया!", "बहुत बढ़िया! बधाई हो। आपकी मेहनत रंग लाई। अब आगे की योजना बनाते हैं?")],
 ('hi','sadness'): [("आज मन बहुत उदास है।", "यह सुनकर मुझे दुख हुआ। आप चाहें तो बताइए क्या हुआ, मैं यहीं हूँ।")],
 ('hi','anger'):   [("मुझे बहुत गुस्सा आ रहा है, कोड बार-बार फेल हो रहा है।", "गहरी साँस लीजिए। चलिए एरर लॉग साथ में देखते हैं, कारण मिल जाएगा।")],
 ('hi','fear'):    [("कल इंटरव्यू है और मुझे डर लग रहा है।", "घबराना स्वाभाविक है। हम मिलकर अभ्यास कर लेते हैं, आप तैयार हो जाएँगे।")],
 ('hi','humor'):   [("कोई जोक सुनाओ।", "बग को रोशनी पसंद है, इसीलिए प्रोग्रामर डार्क मोड में काम करते हैं।")],
 ('te','joy'):     [("నేను చాలా సంతోషంగా ఉన్నాను, ఈ రోజు నాకు ఉద్యోగం వచ్చింది!", "అభినందనలు! మీ కష్టానికి తగిన ఫలితం దక్కింది. ఇప్పుడు తదుపరి ప్రణాళిక వేద్దామా?")],
 ('te','sadness'): [("ఈ రోజు మనసు చాలా బాధగా ఉంది.", "అది విని నాకు బాధగా ఉంది. మీరు చెప్పాలనుకుంటే చెప్పండి, నేను ఇక్కడే ఉన్నాను.")],
 ('te','anger'):   [("నాకు చాలా కోపంగా ఉంది, కోడ్ మళ్లీ మళ్లీ ఫెయిల్ అవుతోంది.", "ఒక్కసారి లోతుగా శ్వాస తీసుకోండి. ఎర్రర్ లాగ్ కలిసి చూద్దాం, కారణం దొరుకుతుంది.")],
 ('te','fear'):    [("రేపు ఇంటర్వ్యూ ఉంది, నాకు భయంగా ఉంది.", "భయపడటం సహజమే. కలిసి ప్రాక్టీస్ చేద్దాం, మీరు సిద్ధంగా ఉంటారు.")],
 ('te','humor'):   [("ఒక జోక్ చెప్పు.", "బగ్‌లకు వెలుతురు ఇష్టం, అందుకే ప్రోగ్రామర్లు డార్క్ మోడ్ వాడతారు.")],
}
def persona_docs(n=4000, seed=SEED):
    rng = random.Random(seed); keys = list(PERSONA)
    out = []
    for _ in range(n):
        lang, emo = rng.choice(keys); u, a = rng.choice(PERSONA[(lang, emo)])
        tag = '<|humor|>' if emo == 'humor' else ('' if emo == 'safety' else f'<|emo:{emo}|>')
        out.append(f"<|system|>{SYSTEM[lang]}<|end|><|lang:{lang}|>{tag}<|user|>{u}<|end|><|assistant|>{a}<|end|>")
    return out

## 5 · SentencePiece tokenizer (BPE, byte-fallback, Indic-aware)

In [ ]:
import sentencepiece as spm
from datasets import load_dataset

CONTROL = (['<|system|>', '<|user|>', '<|assistant|>', '<|end|>', '<|code|>', '<|sec|>', '<|humor|>']
           + [f'<|emo:{e}|>' for e in ['joy','sadness','anger','fear','surprise','love','neutral']]
           + [f'<|lang:{l}|>' for l in ['en','hi','te']] + ['<|think|>', '<|/think|>'])
PAD, UNK, BOS, EOS = 0, 1, 2, 3
SP_PATH = f'{WORK}/tok/vishwamai.model'

def open_stream(cands):
    # returns (iterator of text, name) for the first candidate that loads
    for path, name, ddir, split, fmt in cands:
        try:
            ds = load_dataset(path, name, data_dir=ddir, split=split, streaming=True)
            it = iter(ds); first = next(it)
            def gen(first=first, it=it, fmt=fmt):
                for ex in [first]:
                    t = fmt(ex)
                    if t: yield t
                for ex in it:
                    t = fmt(ex)
                    if t: yield t
            return gen(), path
        except Exception as e:
            print(f"  ✗ {path} ({name or ddir}): {type(e).__name__}: {str(e)[:100]}")
    return None, None

if not os.path.exists(SP_PATH):
    corpus = f'{WORK}/tok/spm_corpus.txt'
    PER_SRC_DOCS = 6000          # sample documents per source for tokenizer training
    with open(corpus, 'w', encoding='utf-8') as f:
        for name, (w, cands) in SOURCES.items():
            gen, used = open_stream(cands)
            if gen is None: print("skip", name); continue
            n = 0
            for t in gen:
                for line in t.replace('\r', '').split('\n'):
                    line = line.strip()
                    if 8 < len(line) < 4000: f.write(line + '\n')
                n += 1
                if n >= PER_SRC_DOCS: break
            print(f"tokenizer corpus <- {name} ({used}): {n} docs")
        for t in persona_docs(500):
            f.write(t + '\n')
    spm.SentencePieceTrainer.train(
        input=corpus, model_prefix=SP_PATH[:-6], vocab_size=VOCAB, model_type='bpe',
        character_coverage=0.9999, byte_fallback=True, split_digits=True,
        pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS,
        user_defined_symbols=CONTROL, normalization_rule_name='nmt_nfkc',
        input_sentence_size=3_000_000, shuffle_input_sentence=True,
        num_threads=os.cpu_count() or 4, train_extremely_large_corpus=False)
sp = spm.SentencePieceProcessor(model_file=SP_PATH)
print("vocab:", sp.get_piece_size())
for s in ["Hello, how are you today?", "आप कैसे हैं? मैं ठीक हूँ।", "మీరు ఎలా ఉన్నారు? నేను బాగున్నాను.", "def add(a, b): return a + b"]:
    ids = sp.encode(s); print(f"{len(s):3d} chars -> {len(ids):3d} tokens  ({len(s)/len(ids):.2f} chars/token)  {s[:40]}")

## 8 · Model — MoE + QK-norm + YaRN RoPE + multi-token prediction

In [ ]:
class RMSNorm(nn.Module):
    @nn.compact
    def __call__(self, x):
        w = self.param('w', nn.initializers.ones, (x.shape[-1],), jnp.float32)
        x32 = x.astype(jnp.float32)
        y = x32 * jax.lax.rsqrt(jnp.mean(x32 * x32, -1, keepdims=True) + 1e-6)
        return (y * w).astype(x.dtype)

def rope_tables(T, hd, theta, scale=1.0, orig_len=1024):
    # Plain RoPE when scale == 1; YaRN (NTK-by-parts + attention temperature) when scale > 1.
    pos_freqs = theta ** (jnp.arange(0, hd, 2, dtype=jnp.float32) / hd)
    inv = 1.0 / pos_freqs
    mscale = 1.0
    if scale > 1.0:
        corr = lambda n_rot: hd * math.log(orig_len / (n_rot * 2 * math.pi)) / (2 * math.log(theta))
        low, high = max(math.floor(corr(32)), 0), min(math.ceil(corr(1)), hd - 1)
        if low == high: high = high + 0.001
        ramp = jnp.clip((jnp.arange(hd // 2, dtype=jnp.float32) - low) / (high - low), 0, 1)
        extrap = 1.0 - ramp
        inv = (1.0 / (scale * pos_freqs)) * (1 - extrap) + inv * extrap
        mscale = 0.1 * math.log(scale) + 1.0
    f = jnp.outer(jnp.arange(T, dtype=jnp.float32), inv)
    return jnp.cos(f) * mscale, jnp.sin(f) * mscale

def apply_rope(x, cos, sin):                      # x: [B,T,H,D]; half-split layout == HF rotate_half
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    c, s = cos[None, :, None, :].astype(x.dtype), sin[None, :, None, :].astype(x.dtype)
    return jnp.concatenate([x1 * c - x2 * s, x1 * s + x2 * c], -1)

def dense(cfg, n, name, scale=1.0):
    return nn.Dense(n, use_bias=False, dtype=cfg.dtype, param_dtype=jnp.float32, name=name,
                    kernel_init=nn.initializers.normal(0.02 * scale))

class Attn(nn.Module):
    cfg: Cfg
    @nn.compact
    def __call__(self, x, cos, sin):
        c = self.cfg; hd = c.d_model // c.n_heads; B, T, _ = x.shape
        q = dense(c, c.n_heads * hd, 'q')(x).reshape(B, T, c.n_heads, hd)
        k = dense(c, c.n_kv_heads * hd, 'k')(x).reshape(B, T, c.n_kv_heads, hd)
        v = dense(c, c.n_kv_heads * hd, 'v')(x).reshape(B, T, c.n_kv_heads, hd)
        q, k = RMSNorm(name='q_norm')(q), RMSNorm(name='k_norm')(k)          # QK-norm (per head, before RoPE)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        r = c.n_heads // c.n_kv_heads
        k, v = jnp.repeat(k, r, axis=2), jnp.repeat(v, r, axis=2)
        o = jax.nn.dot_product_attention(q, k, v, is_causal=True)
        return dense(c, c.d_model, 'o', scale=1 / math.sqrt(2 * c.n_layers))(o.reshape(B, T, -1))

class MLP(nn.Module):                              # dense SwiGLU (used by the MTP blocks)
    cfg: Cfg
    @nn.compact
    def __call__(self, x):
        c = self.cfg
        h = nn.silu(dense(c, c.d_ff, 'gate')(x)) * dense(c, c.d_ff, 'up')(x)
        return dense(c, c.d_model, 'down', scale=1 / math.sqrt(2 * c.n_layers))(h)

class MoE(nn.Module):
    # Top-k routed SwiGLU experts. Routing is per sequence (vmap over batch) so it stays local to a device;
    # tokens are scattered into [E, capacity, d] buffers (no giant one-hot dispatch tensor).
    cfg: Cfg
    @nn.compact
    def __call__(self, x):                         # x: [B,T,d] -> (y, aux_loss)
        c = self.cfg; B, T, d = x.shape; E, K, dt = c.n_experts, c.top_k, c.dtype
        init = nn.initializers.normal(0.02)
        router = self.param('router', init, (d, E), jnp.float32)
        wg = self.param('wg', init, (E, d, c.moe_ff), jnp.float32)
        wu = self.param('wu', init, (E, d, c.moe_ff), jnp.float32)
        wd = self.param('wd', nn.initializers.normal(0.02 / math.sqrt(2 * c.n_layers)), (E, c.moe_ff, d), jnp.float32)
        C = min(T * K, int(math.ceil(c.cap_factor * T * K / E)))
        def one(xs):                               # xs: [T,d]
            logits = xs.astype(jnp.float32) @ router
            probs = jax.nn.softmax(logits, -1)
            topw, topi = jax.lax.top_k(probs, K)
            topw = topw / topw.sum(-1, keepdims=True)           # normalise top-k weights
            fe, fw, ft = topi.reshape(-1), topw.reshape(-1), jnp.repeat(jnp.arange(T), K)
            oh = jax.nn.one_hot(fe, E, dtype=jnp.int32)
            pos = jnp.take_along_axis(jnp.cumsum(oh, 0) - 1, fe[:, None], 1)[:, 0]   # slot inside the expert
            keep = pos < C
            slot = jnp.where(keep, fe * C + pos, E * C)
            tok = jnp.zeros(E * C + 1, jnp.int32).at[slot].set(ft)
            val = jnp.zeros(E * C + 1, jnp.float32).at[slot].set(1.0)
            xe = (xs[tok[:E * C]] * val[:E * C, None].astype(xs.dtype)).reshape(E, C, d)
            h = nn.silu(jnp.einsum('ecd,edf->ecf', xe, wg.astype(dt))) * jnp.einsum('ecd,edf->ecf', xe, wu.astype(dt))
            ye = jnp.einsum('ecf,efd->ecd', h, wd.astype(dt)).reshape(E * C, d)
            ya = jnp.where(keep[:, None], ye[jnp.minimum(slot, E * C - 1)], 0) * fw[:, None].astype(dt)
            f = oh.sum(0).astype(jnp.float32) / (T * K)                      # routed fraction (pre-drop)
            lb = E * jnp.sum(f * probs.mean(0))                              # Switch load-balance loss
            z = jnp.mean(jax.nn.logsumexp(logits, -1) ** 2)                  # router z-loss
            return ya.reshape(T, K, d).sum(1), lb, z
        y, lb, z = jax.vmap(one)(x)
        return y, c.aux_coef * lb.mean() + c.router_z * z.mean()

class Block(nn.Module):
    cfg: Cfg
    moe: bool = True
    @nn.compact
    def __call__(self, x, cos, sin):
        x = x + Attn(self.cfg, name='attn')(RMSNorm(name='n1')(x), cos, sin)
        h = RMSNorm(name='n2')(x)
        if self.moe:
            y, aux = MoE(self.cfg, name='moe')(h)
        else:
            y, aux = MLP(self.cfg, name='mlp')(h), jnp.zeros((), jnp.float32)
        return x + y, aux

class VishwamAI(nn.Module):
    cfg: Cfg
    @nn.compact
    def __call__(self, ids, train_mtp=False):      # -> (logits, [mtp_logits...], aux_loss)
        c = self.cfg; B, T = ids.shape
        emb = nn.Embed(c.vocab, c.d_model, dtype=c.dtype, param_dtype=jnp.float32,
                       embedding_init=nn.initializers.normal(0.02), name='embed')
        head = lambda h: h.astype(jnp.float32) @ emb.embedding.astype(jnp.float32).T      # tied output head
        x = emb(ids)
        cos, sin = rope_tables(T, c.d_model // c.n_heads, c.rope_theta, c.rope_scale, c.rope_orig_len)
        RBlock = nn.remat(Block)                   # activation checkpointing
        aux = jnp.zeros((), jnp.float32)
        for i in range(c.n_layers):
            x, a = RBlock(c, True, name=f'block_{i}')(x, cos, sin); aux = aux + a
        logits = head(RMSNorm(name='norm_f')(x))
        mtp, hp = [], x
        if train_mtp and c.mtp_depth > 0:          # multi-token prediction: head j predicts token t+j+1
            for j in range(1, c.mtp_depth + 1):
                e = jnp.concatenate([emb(ids)[:, j:], jnp.zeros((B, j, c.d_model), c.dtype)], 1)
                z = dense(c, c.d_model, f'mtp{j}_proj')(jnp.concatenate(
                    [RMSNorm(name=f'mtp{j}_nh')(hp), RMSNorm(name=f'mtp{j}_ne')(e)], -1))
                hp, _ = Block(c, False, name=f'mtp{j}_block')(z, cos, sin)
                mtp.append(head(RMSNorm(name=f'mtp{j}_nf')(hp)))
        return logits, mtp, aux

def count_params(params, c):
    flat = jax.tree_util.tree_flatten_with_path(params)[0]
    key = lambda path: '/'.join(str(getattr(k, 'key', k)) for k in path)
    tot = sum(v.size for _, v in flat)
    mtp = sum(v.size for p, v in flat if 'mtp' in key(p))
    exp = sum(v.size for p, v in flat if key(p).split('/')[-1] in ('wg', 'wu', 'wd'))
    total_inf = tot - mtp
    active = total_inf - exp * (1 - c.top_k / c.n_experts)
    return total_inf, active, mtp

model_init = VishwamAI(cfg)
params = model_init.init(jax.random.PRNGKey(SEED), jnp.zeros((1, 16), jnp.int32), train_mtp=True)['params']
total_p, active_p, mtp_p = count_params(params, cfg)
print(f"{PRESET}: {total_p/1e6:.1f}M total | {active_p/1e6:.1f}M active per token | +{mtp_p/1e6:.1f}M training-only MTP heads")

## 9 · Training machinery (pre-training, SFT, DPO, GRPO, checkpoints)

In [ ]:
import zlib
devices = jax.devices(); nd = len(devices)
mesh = Mesh(np.array(devices), ('data',))
REPL, ROW, ACC_ROW = NamedSharding(mesh, P()), NamedSharding(mesh, P('data')), NamedSharding(mesh, P(None, 'data'))

# ---------- checkpoints ----------
def save_tree(path, tree):
    tmp = path + '.tmp'; open(tmp, 'wb').write(fser.to_bytes(jax.device_get(tree))); os.replace(tmp, path)
def load_tree(path, like):
    return fser.from_bytes(jax.device_get(like), open(path, 'rb').read())
def ckpts(tag): return sorted(glob.glob(f'{WORK}/ckpt/{tag}_step_*.msgpack'))
def final_path(tag): return f'{WORK}/ckpt/{tag}_final.msgpack'

# ---------- losses ----------
def ce_terms(logits, y, m):
    m = m.astype(jnp.float32)
    logz = jax.nn.logsumexp(logits, -1)
    nll = logz - jnp.take_along_axis(logits, y[..., None], -1)[..., 0]
    n = jnp.maximum(m.sum(), 1.0)
    return (nll * m).sum() / n, ((logz ** 2) * m).sum() / n

def make_pretrain_loss(mdl, c):
    def loss_fn(p, batch):                                   # batch: [B, T+1]
        x, y = batch[:, :-1], batch[:, 1:]
        logits, mtp, aux = mdl.apply({'params': p}, x, train_mtp=True)
        main, z = ce_terms(logits, y, y != PAD)
        total = main + Z_LOSS * z + aux
        mt = jnp.zeros(())
        for j, lg in enumerate(mtp, 1):                      # head j predicts token t+j+1
            yj = jnp.pad(batch[:, j + 1:], ((0, 0), (0, j)))
            l, _ = ce_terms(lg, yj, yj != PAD); mt = mt + l
        if mtp: total = total + MTP_W * mt / len(mtp)
        return total, (main, aux, mt)
    return loss_fn

def make_fwd(c):                                              # dropless inference forward -> logits
    m = VishwamAI(eval_cfg(c))
    return jax.jit(lambda p, x: m.apply({'params': p}, x)[0])

def make_eval_nll(c):
    m = VishwamAI(eval_cfg(c))
    @jax.jit
    def f(p, batch):
        logits, _, _ = m.apply({'params': p}, batch[:, :-1]); y = batch[:, 1:]
        return ce_terms(logits, y, y != PAD)[0]
    return f

def sample_text(fwd, params, prompt_ids, max_new=120, temperature=0.8, top_p=0.9, seed=0, stop=(), buf_len=256):
    ids = list(prompt_ids); buf = np.zeros((1, buf_len), np.int32); buf[0, :len(ids)] = ids; n = len(ids)
    r = np.random.default_rng(seed)
    for _ in range(min(max_new, buf_len - n)):
        lg = np.asarray(fwd(params, jnp.asarray(buf))[0, n - 1], dtype=np.float64) / max(temperature, 1e-4)
        pr = np.exp(lg - lg.max()); pr /= pr.sum()
        o = np.argsort(-pr); cs = np.cumsum(pr[o]); keep = o[:max(1, int((cs < top_p).sum()) + 1)]
        nxt = int(r.choice(keep, p=pr[keep] / pr[keep].sum()))
        if nxt == EOS or nxt in stop: break
        buf[0, n] = nxt; n += 1
    return buf[0, len(prompt_ids):n].tolist()

# ---------- pre-training stage (also used for the YaRN context-extension stage) ----------
def run_pretrain(tag, c, seq_len, token_budget, lr, warmup, params):
    fin = final_path(tag)
    if os.path.exists(fin):
        print(f'[{tag}] already finished -> loading'); return jax.device_put(load_tree(fin, params), REPL)
    mdl = VishwamAI(c)
    mg = nd * max(1, PER_DEVICE_TOKENS // seq_len)                   # sequences per micro-step (global)
    accum = max(1, GLOBAL_TOKENS_PER_STEP // (mg * seq_len)); tok_step = accum * mg * seq_len
    total = max(2, int(token_budget // tok_step))
    sched = optax.warmup_cosine_decay_schedule(0.0, lr, min(warmup, max(1, total // 10)), total, lr * MIN_LR_FRAC)
    tx = optax.chain(optax.clip_by_global_norm(1.0),
                     optax.adamw(sched, b1=0.9, b2=0.95, eps=1e-8, weight_decay=WD,
                                 mask=lambda p: jax.tree.map(lambda x: x.ndim >= 2, p)))
    loss_fn = make_pretrain_loss(mdl, c)
    @partial(jax.jit, donate_argnums=(0, 1))
    def step_fn(p, o, batch):                                        # batch: [accum, B, T+1]
        def micro(carry, mb):
            g_acc, m_acc = carry
            (_, (main, aux, mt)), g = jax.value_and_grad(loss_fn, has_aux=True)(p, mb)
            return (jax.tree.map(jnp.add, g_acc, g), m_acc + jnp.stack([main, aux, mt])), None
        (g, m), _ = jax.lax.scan(micro, (jax.tree.map(jnp.zeros_like, p), jnp.zeros(3)), batch)
        g = jax.tree.map(lambda x: x / batch.shape[0], g)
        upd, o = tx.update(g, o, p)
        return optax.apply_updates(p, upd), o, m / batch.shape[0], optax.global_norm(g)
    params = jax.device_put(params, REPL); opt = jax.device_put(tx.init(params), REPL); step0 = 0
    ck = ckpts(tag)
    if ck:
        st = load_tree(ck[-1], {'params': params, 'opt': opt, 'step': 0})
        params, opt, step0 = jax.device_put(st['params'], REPL), jax.device_put(st['opt'], REPL), int(st['step'])
        print(f'[{tag}] resumed at step {step0}')
    salt = zlib.crc32(tag.encode()) % 100_000
    def get_batch(step):
        r = np.random.default_rng(SEED * 1_000_003 + salt * 7919 + step)       # deterministic -> exact resume
        b = sample_batch(r, accum * mg, seq_len).reshape(accum, mg, seq_len + 1)
        return jax.device_put(b, ACC_ROW)
    eval_nll, fwd = make_eval_nll(c), make_fwd(c)
    def do_eval():
        r = np.random.default_rng(999); out = {}
        for n in names:
            if len(shards[n]['val']) < seq_len + 2: continue
            out[n] = float(eval_nll(params, jax.device_put(sample_batch(r, mg, seq_len, 'val', only=n), ROW)))
        return out
    print(f'[{tag}] {nd} device(s) | seq {seq_len} | {tok_step/1e3:.0f}k tok/step | {total} steps | rope_scale {c.rope_scale}')
    t0, seen = time.time(), 0
    for step in range(step0, total):
        params, opt, m, gn = step_fn(params, opt, get_batch(step)); seen += tok_step
        if (step + 1) % LOG_EVERY == 0 or step == step0 + 4:
            main, aux, mt = [float(v) for v in m]; tps = seen / (time.time() - t0)
            eta = (total - step - 1) * tok_step / tps / 3600
            print(f'[{tag}] {step+1}/{total} nll {main:.3f} ppl {math.exp(main):.1f} aux {aux:.4f} mtp {mt:.3f} '
                  f'gnorm {float(gn):.2f} {tps/1e3:.1f}k tok/s ETA {eta:.1f}h')
            if not math.isfinite(main): raise RuntimeError('loss diverged — lower LR / check data')
        if (step + 1) % min(EVAL_EVERY, max(5, total // 6)) == 0:
            print('  val nll:', {k: round(v, 3) for k, v in do_eval().items()})
            for pr in ['The capital of India is', 'भारत की राजधानी', 'తెలుగు భాష']:
                print('  >', sp.decode(sample_text(fwd, params, [EOS] + sp.encode(pr), 40)).replace('\n', ' ')[:140])
        if (step + 1) % min(CKPT_EVERY, max(5, total // 10)) == 0:      # short runs checkpoint more often
            save_tree(f'{WORK}/ckpt/{tag}_step_{step+1:07d}.msgpack', {'params': params, 'opt': opt, 'step': step + 1})
            for old in ckpts(tag)[:-2]: os.remove(old)
    save_tree(fin, params)
    for old in ckpts(tag): os.remove(old)
    print(f'[{tag}] finished'); return params

## 10 · Stage 1 — pre-training (1k context)
Safe to interrupt and re-run any cell: shards and checkpoints live on Drive, batches are a pure function of the step number, and a finished stage is detected and skipped.
**Time estimate:** FLOPs ≈ 6 · (*active* params) · tokens, but MoE runs at roughly half the utilisation of a dense model (routing + padded expert buffers). The log prints a live ETA after the first steps; with `base` (≈40M active) on an A100 or TPU v2-8, 2B tokens is on the order of 8–20 h.

In [ ]:
params = run_pretrain('pt', cfg, SEQ_LEN, TOKEN_BUDGET, LR, WARMUP, params)

## 11 · Stage 2 — long-context extension (1k → 4k) with YaRN
Same weights, new RoPE frequencies (YaRN scale 4 + attention temperature), trained on 4k windows for ~3 % of the token budget. From here on **every** later stage and the export use `cfg_ext`.

In [ ]:
params = run_pretrain('ctx', cfg_ext, CTX_LEN, CTX_TOKENS, LR * 0.3, 50, params)
CFG = cfg_ext                      # final architecture config used from now on
FWD = make_fwd(CFG)

## 12 · Chat format, reasoning tasks and generation helpers
Format: `<|system|>…<|end|><|lang:xx|>[<|emo:…|>|<|humor|>]<|user|>…<|end|><|assistant|><|think|>…<|/think|>answer<|end|>`. Loss is applied **only** to assistant tokens.
Reasoning data is human-written (GSM8K train solutions) or **programmatically generated with exact answers** (arithmetic, graph walks) — no other model writes any training text.

In [ ]:
ASSIST, END = sp.piece_to_id('<|assistant|>'), sp.piece_to_id('<|end|>')

def render(system, turns, lang=None, tag=''):
    s = f"<|system|>{system}<|end|>" + (f"<|lang:{lang}|>" if lang else '') + tag
    for role, t in turns:
        s += f"<|user|>{t}<|end|>" if role == 'user' else f"<|assistant|>{t}<|end|>"
    return s

def encode_chat(text, max_len):
    ids = [EOS] + sp.encode(text); mk = [0] * len(ids); on = False
    for i, t in enumerate(ids):
        if on: mk[i] = 1
        if t == ASSIST: on = True
        elif t == END and on: on = False
    return ids[:max_len + 1], mk[:max_len + 1]

def prompt_ids(user, lang='en', tag='', system=None, history=()):
    return [EOS] + sp.encode(render(system or SYSTEM[lang], list(history) + [('user', user)], lang, tag) + '<|assistant|>')

def tokenize_examples(texts, L):
    ids = np.zeros((len(texts), L + 1), np.int32); mk = np.zeros((len(texts), L + 1), np.uint8)
    for i, t in enumerate(texts):
        a, m = encode_chat(t, L); ids[i, :len(a)] = a; mk[i, :len(m)] = m
    keep = mk[:, 1:].sum(1) > 0
    return ids[keep], mk[keep]

# ---------- programmatic reasoning tasks (exact answers) ----------
ARITH_Q = {'en': "Compute: {e}", 'hi': "गणना कीजिए: {e}", 'te': "లెక్కించండి: {e}"}
def arith_problem(rng, lang, n_ops):
    val = rng.randint(2, 50 if n_ops > 1 else 99); expr = str(val); steps = []
    for k in range(n_ops):
        op = rng.choice('+-*'); b = rng.randint(2, 12 if op == '*' else 50)
        new = val + b if op == '+' else val - b if op == '-' else val * b
        steps.append(f"{val} {op} {b} = {new}")
        expr = f"{expr} {op} {b}" if k == 0 else f"({expr}) {op} {b}"
        val = new
    return ARITH_Q[lang].format(e=expr), "<|think|>" + "\n".join(steps) + f"<|/think|>#### {val}", str(val)

def extract_answer(text):
    m = re.findall(r'####\s*(-?[\d,]*\.?\d+)', text)
    return m[-1].replace(',', '') if m else None

def same_number(a, b):
    try: return abs(float(a) - float(b)) < 1e-6
    except Exception: return False

def graphwalks_synth(rng, n_nodes, n_edges, depth):
    nodes = [f"n{i}" for i in range(n_nodes)]; edges = set()
    while len(edges) < n_edges:
        a, b = rng.sample(nodes, 2); edges.add((a, b))
    edges = sorted(edges); rng.shuffle(edges)
    adj = {}
    for a, b in edges: adj.setdefault(a, []).append(b)
    start = rng.choice(nodes); seen, frontier = {start}, [start]
    for _ in range(depth):
        nxt = []
        for u in frontier:
            for v in adj.get(u, []):
                if v not in seen: seen.add(v); nxt.append(v)
        frontier = sorted(set(nxt))
    q = ("Graph edges (a -> b):\n" + "\n".join(f"{a} -> {b}" for a, b in edges) +
         f"\nPerform a BFS from {start} with depth {depth}. List the nodes first reached at exactly depth {depth}.")
    return q, "#### [" + ", ".join(frontier) + "]", frontier

def f1_set(pred, gold):
    pred, gold = set(pred), set(gold)
    if not pred and not gold: return 1.0
    tp = len(pred & gold)
    if tp == 0: return 0.0
    p, r = tp / len(pred), tp / len(gold)
    return 2 * p * r / (p + r)

# ---------- batched greedy / sampled generation (re-forward; fine for small models) ----------
_SAMPLERS = {}
def make_sampler(c, buf_len, max_new):
    key_ = (c, buf_len, max_new)
    if key_ in _SAMPLERS: return _SAMPLERS[key_]
    m = VishwamAI(eval_cfg(c))
    @jax.jit
    def gen(p, buf, plen, key, temp):
        N = buf.shape[0]
        def body(i, carry):
            b, k = carry
            logits = m.apply({'params': p}, b)[0]
            pos = plen + i
            lg = jnp.take_along_axis(logits, (pos - 1)[:, None, None], 1)[:, 0]
            k, sk = jax.random.split(k)
            tok = jnp.where(temp > 0, jax.random.categorical(sk, lg / jnp.maximum(temp, 1e-4)),
                            jnp.argmax(lg, -1)).astype(jnp.int32)
            return b.at[jnp.arange(N), pos].set(tok), k
        b, _ = jax.lax.fori_loop(0, max_new, body, (buf, key))
        return b
    _SAMPLERS[key_] = gen
    return gen

def generate(c, params, prompts, max_new=64, temp=0.0, seed=0, bs=None, buf_len=None):
    # prompts: list of token-id lists -> list of completion token-id lists (cut at <|end|>)
    bs = bs or 4 * nd
    buf_len = buf_len or int(math.ceil((max(len(p) for p in prompts) + max_new) / 16) * 16)
    gen = make_sampler(c, buf_len, max_new); outs = []
    for s in range(0, len(prompts), bs):
        chunk = prompts[s:s + bs]; n_real = len(chunk)
        chunk = chunk + [chunk[-1]] * (bs - n_real)
        buf = np.zeros((bs, buf_len), np.int32); plen = np.zeros(bs, np.int32)
        for i, p in enumerate(chunk): buf[i, :len(p)] = p; plen[i] = len(p)
        res = np.asarray(gen(params, jnp.asarray(buf), jnp.asarray(plen), jax.random.PRNGKey(seed + s), jnp.float32(temp)))
        for i in range(n_real):
            tail = res[i, plen[i]:plen[i] + max_new].tolist()
            outs.append(tail[:tail.index(END) + 1] if END in tail else tail)
    return outs

## 13 · Benchmark suite
**What "covering" those benchmarks honestly means.** Frontier scores (60–99 %) come from models with 100–1000× more parameters *and* agent scaffolds (tools, sandboxes, browsers). This notebook gives you: (a) benchmarks it can **actually run now** (MMLU, MMMLU-Hindi, GSM8K, GraphWalks, in-house reasoning), (b) a **registry** showing, per benchmark, the dataset and exactly what is missing to run it (agent harness, VM, vision, >100k context, or the benchmark isn't public). Expect near-zero on the agentic rows until the model is much larger and has an agent stage.

In [ ]:
from datasets import load_dataset
LETTER_IDS = [sp.piece_to_id('▁' + c_) for c_ in 'ABCD']
assert all(i != UNK for i in LETTER_IDS), LETTER_IDS

def mc_prompt(q, choices, ans=None):
    s = q.strip() + "\n" + "\n".join(f"{l}. {c_}" for l, c_ in zip('ABCD', choices)) + "\nAnswer:"
    return s + (f" {'ABCD'[ans]}\n\n" if ans is not None else "")

CHAT_LETTER_IDS = [sp.encode(f'<|assistant|>{l_}')[-1] for l_ in 'ABCD']     # same tokenisation path as SFT targets

def make_choice_fn(c, letter_ids=None):
    m = VishwamAI(eval_cfg(c)); lid = jnp.array(letter_ids or LETTER_IDS)
    @jax.jit
    def f(p, ids, last):
        lg = m.apply({'params': p}, ids)[0]
        return jnp.take_along_axis(lg, last[:, None, None], 1)[:, 0, :][:, lid]
    return f

def eval_mc(c, params, rows, shots_rows=(), max_n=1000, bs=8, L=SEQ_LEN, chat=False):
    choice_fn = make_choice_fn(c, CHAT_LETTER_IDS if chat else None); correct = 0; batch = []
    def flush():
        nonlocal correct, batch
        n_real = len(batch)
        while len(batch) < bs: batch.append(batch[-1])
        pad = np.zeros((bs, L), np.int32); last = np.zeros(bs, np.int32)
        for i, (ids, _) in enumerate(batch): pad[i, :len(ids)] = ids; last[i] = len(ids) - 1
        pred = np.asarray(choice_fn(params, jnp.asarray(pad), jnp.asarray(last))).argmax(-1)
        correct += sum(int(pred[i] == batch[i][1]) for i in range(n_real)); batch = []
    rows = rows[:max_n]
    for q, ch, a in rows:
        head = ""
        for sq, sc, sa in shots_rows:
            cand = head + mc_prompt(sq, sc, sa)
            if len(sp.encode(cand + mc_prompt(q, ch))) < L - 4: head = cand
        ids_ = prompt_ids(mc_prompt(q, ch), 'en') if chat else [EOS] + sp.encode(head + mc_prompt(q, ch))
        batch.append((ids_[-L:], a))
        if len(batch) == bs: flush()
    if batch: flush()
    return correct / len(rows)

def eval_arith(c, params, n=128, n_ops=(1, 2, 3), seed=7):
    rng = random.Random(seed); out = {}
    for k in n_ops:
        items = [arith_problem(rng, rng.choice(['en', 'hi', 'te']), k) for _ in range(n)]
        comp = generate(c, params, [prompt_ids(q, 'en') for q, _, _ in items], max_new=64, temp=0.0)
        out[k] = float(np.mean([same_number(extract_answer(sp.decode(x)), a) for x, (_, _, a) in zip(comp, items)]))
    return out

def eval_gsm8k(c, params, n=200):
    ds = load_dataset('openai/gsm8k', 'main', split='test').select(range(n))
    comp = generate(c, params, [prompt_ids(r['question'], 'en') for r in ds], max_new=256, temp=0.0)
    gold = [r['answer'].split('####')[-1].strip().replace(',', '') for r in ds]
    return float(np.mean([same_number(extract_answer(sp.decode(x)), g) for x, g in zip(comp, gold)]))

def eval_graphwalks(c, params, sizes=(10, 40, 150, 400), n=16, seed=3):
    rng = random.Random(seed); out = {}
    for ne in sizes:
        items = [graphwalks_synth(rng, max(8, int(ne * 0.7)), ne, rng.choice([1, 2])) for _ in range(n)]
        prompts = [prompt_ids(q, 'en') for q, _, _ in items]
        if max(map(len, prompts)) + 64 > c.max_len: out[ne] = None; continue
        comp = generate(c, params, prompts, max_new=48, temp=0.0, bs=nd)
        out[ne] = float(np.mean([f1_set(re.findall(r'n\d+', sp.decode(x).split('####')[-1]), g)
                                 for x, (_, _, g) in zip(comp, items)]))
    return out

def eval_all(c, params, tag=''):
    res = {}
    try:
        mmlu = load_dataset('cais/mmlu', 'all')
        dev = [(r['question'], r['choices'], r['answer']) for r in mmlu['dev']][:5]
        test = [(r['question'], r['choices'], r['answer']) for r in mmlu['test']]; random.Random(0).shuffle(test)
        res['MMLU 5-shot (raw format)'] = eval_mc(c, params, test, dev)
        res['MMLU 0-shot (chat format)'] = eval_mc(c, params, test, chat=True)
    except Exception as e: print('MMLU skipped:', e)
    try:
        hi = load_dataset('openai/MMMLU', 'HI_IN', split='test')
        rows = [(r['Question'], [r['A'], r['B'], r['C'], r['D']], 'ABCD'.index(r['Answer'])) for r in hi]; random.Random(0).shuffle(rows)
        res['MMMLU-Hindi 0-shot (raw)'] = eval_mc(c, params, rows)
        res['MMMLU-Hindi 0-shot (chat)'] = eval_mc(c, params, rows, chat=True)
    except Exception as e: print('MMMLU skipped:', e)
    try: res['GSM8K (test, greedy)'] = eval_gsm8k(c, params)
    except Exception as e: print('GSM8K skipped:', e)
    res['arithmetic acc by #ops'] = eval_arith(c, params)
    res['GraphWalks-synthetic F1 by #edges'] = eval_graphwalks(c, params)
    print(f'===== {tag} ====='); [print(f'{k:36s} {v}') for k, v in res.items()]
    return res

### Baseline on the imported base model
Save these numbers: they are what distillation has to beat.

In [ ]:
# Baseline after pre-training (expect ~chance; this is the regression test for later stages)
res_pt = eval_all(CFG, params, 'after pre-training')

## Distillation data (local open-weight teachers)
### Prompts
Verifiable math (programmatic arithmetic + GSM8K **train**) in en/hi/te plus general chat prompts. Math prompts carry the known answer, so teacher replies with a wrong final number are rejected.

In [ ]:
MATH_SUFFIX = "\n\nEnd your reply with a final line '#### <number>'."
N_ARITH, N_GSM8K, N_CHAT, N_INDIC = 3000, 3000, 3000, 1500        # prompts per source
os.makedirs(f'{WORK}/distill', exist_ok=True)
rng = random.Random(SEED + 7); rows = []

for _ in range(N_ARITH):                                           # verifiable arithmetic, en/hi/te
    lang = rng.choice(['en', 'hi', 'te']); q, _, ans = arith_problem(rng, lang, rng.choice([1, 2, 2, 3]))
    rows.append(dict(prompt=q + MATH_SUFFIX, lang=lang, answer=ans))
for k, r in enumerate(load_dataset('openai/gsm8k', 'main', split='train')):   # train split only
    if k >= N_GSM8K: break
    rows.append(dict(prompt=r['question'] + MATH_SUFFIX, lang='en',
                     answer=r['answer'].split('####')[-1].strip().replace(',', '')))
for k, r in enumerate(load_dataset('databricks/databricks-dolly-15k', split='train')):
    if k >= N_CHAT: break
    rows.append(dict(prompt=r['instruction'] + (f"\n\n{r['context']}" if r['context'] else ''), lang='en'))
cnt = {'hin': 0, 'tel': 0}
for r in load_dataset('CohereLabs/aya_dataset', split='train'):    # Hindi / Telugu chat prompts
    k = r['language_code']
    if k in cnt and cnt[k] < N_INDIC:
        cnt[k] += 1; rows.append(dict(prompt=r['inputs'], lang={'hin': 'hi', 'tel': 'te'}[k]))
rng.shuffle(rows)
with open(f'{WORK}/distill/prompts.jsonl', 'w', encoding='utf-8') as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(len(rows), 'prompts ->', f'{WORK}/distill/prompts.jsonl')

### Teacher generator
Writes `distill_generate.py` (same file as in the repo). It loads the safetensors weights with `transformers`, reads DeepSeek's `<think>` trace / gpt-oss's analysis channel as reasoning, and does rejection sampling on math answers.

In [ ]:
%%writefile distill_generate.py
"""Sequence-level distillation data generator for VishwamAI (local weights, no API).

Loads an open-weight teacher from safetensors with transformers and writes JSONL
ready to merge into the Stage-3 SFT data of vishwamai_pretrain.ipynb.

Teachers (override the repo with --model):
    deepseek  deepseek-ai/DeepSeek-R1-Distill-Qwen-7B   (the full 671B R1 does not fit one GPU)
    gptoss    openai/gpt-oss-20b                        (Apache 2.0, needs a recent transformers)
    gemma     google/gemma-2-9b-it                      (gated: accept the licence, set HF_TOKEN)

Usage:
    python distill_generate.py --teacher deepseek --prompts prompts.jsonl \
        --out out_deepseek.jsonl --limit 20000 --batch 16 [--4bit]

prompts.jsonl: one object per line: {"prompt": str, "lang": "en|hi|te",
"answer": optional verifiable final answer}.

Output line: {"system", "lang", "prompt", "think", "answer", "teacher"}
mapping onto the notebook's chat format
    <|user|>..<|end|><|assistant|><|think|>{think}<|/think|>{answer}<|end|>
Check each teacher's licence before releasing a model trained on its output.
"""
import argparse, json, re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

TEACHERS = {
    'deepseek': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
    'gptoss':   'openai/gpt-oss-20b',
    'gemma':    'google/gemma-2-9b-it',
}
SYSTEM = {
    'en': 'You are VishwamAI, a calm, witty and emotionally aware assistant. '
          'Reason briefly step by step, then give the final answer.',
    'hi': 'आप विश्वमAI हैं, एक शांत और संवेदनशील सहायक। पहले संक्षेप में चरण-दर-चरण सोचें, फिर उत्तर दें।',
    'te': 'మీరు విశ్వమAI, ప్రశాంతమైన సహాయకుడు. ముందు క్లుప్తంగా ఆలోచించి, తర్వాత సమాధానం ఇవ్వండి.',
}


def final_number(s):
    m = re.findall(r'-?\d[\d,]*\.?\d*', s or '')
    return m[-1].replace(',', '').rstrip('.') if m else None


def split_reasoning(teacher, raw):
    """raw is decoded with special tokens kept; returns (think, answer)."""
    if teacher == 'deepseek':                       # R1-distill: <think>...</think>answer
        raw = re.sub(r'<｜[^>]*｜>', '', raw)
        if '</think>' in raw:
            think, ans = raw.split('</think>', 1)
            return think.replace('<think>', '').strip(), ans.strip()
        return '', raw.strip()
    if teacher == 'gptoss':                         # harmony: analysis channel, then final
        m = re.search(r'<\|channel\|>analysis<\|message\|>(.*?)<\|end\|>', raw, re.S)
        f = re.search(r'<\|channel\|>final<\|message\|>(.*?)(?:<\|return\|>|<\|end\|>|$)', raw, re.S)
        return (m.group(1).strip() if m else ''), (f.group(1).strip() if f else '')
    raw = re.sub(r'<[^>]*>', '', raw)               # gemma: no native reasoning trace
    parts = re.split(r'(?i)\bfinal answer\s*:', raw, maxsplit=1)
    return (parts[0].strip(), parts[1].strip()) if len(parts) == 2 else ('', raw.strip())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--teacher', choices=TEACHERS, required=True)
    ap.add_argument('--model', help='override the Hugging Face repo or a local path')
    ap.add_argument('--prompts', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--limit', type=int, default=1000)
    ap.add_argument('--batch', type=int, default=8)
    ap.add_argument('--max-new', type=int, default=1024)
    ap.add_argument('--4bit', dest='four_bit', action='store_true', help='bitsandbytes 4-bit (fits a T4)')
    a = ap.parse_args()

    name = a.model or TEACHERS[a.teacher]
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(device_map='auto', torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    if a.four_bit:
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()

    exs = [json.loads(l) for l in open(a.prompts, encoding='utf-8')][:a.limit]
    kept = dropped = 0
    with open(a.out, 'a', encoding='utf-8') as f:
        for i in range(0, len(exs), a.batch):
            chunk = exs[i:i + a.batch]
            texts = []
            for ex in chunk:
                msgs = [{'role': 'user', 'content': SYSTEM.get(ex.get('lang', 'en'), SYSTEM['en'])
                         + '\n\n' + ex['prompt']}]    # system folded into user: Gemma has no system role
                texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=a.max_new, do_sample=True,
                                     temperature=0.6, top_p=0.95)
            for ex, ids in zip(chunk, out[:, enc['input_ids'].shape[1]:]):
                think, answer = split_reasoning(a.teacher, tok.decode(ids, skip_special_tokens=False))
                for s in (tok.eos_token, tok.pad_token):
                    answer = answer.replace(s or '', '').strip()
                if not answer:
                    dropped += 1; continue
                # rejection sampling: keep only answers matching a known ground truth
                if ex.get('answer') is not None and final_number(answer) != str(ex['answer']):
                    dropped += 1; continue
                lang = ex.get('lang', 'en')
                f.write(json.dumps(dict(system=SYSTEM.get(lang, SYSTEM['en']), lang=lang,
                                        prompt=ex['prompt'], think=think, answer=answer,
                                        teacher=a.teacher), ensure_ascii=False) + '\n')
                kept += 1
            f.flush()
            print(f'{i + len(chunk)}/{len(exs)} kept {kept} dropped {dropped}', flush=True)


if __name__ == '__main__':
    main()

### Generate

In [ ]:
# One teacher at a time (each loads its own weights from the Hub, then frees the GPU).
# Resumable: outputs are appended per batch; delete a file to regenerate it.
TEACHERS_RUN = {            # teacher: number of prompts
    'deepseek': 8000,       # R1-style reasoning traces
    'gptoss':   4000,       # needs a recent transformers; ~16 GB even quantised on some stacks
    'gemma':    4000,       # gated: accept the licence on the Hub first
}
BATCH, USE_4BIT = 16, True
!pip -q install -U transformers accelerate bitsandbytes
for t, n in TEACHERS_RUN.items():
    out = f'{WORK}/distill/out_{t}.jsonl'
    done = sum(1 for _ in open(out, encoding='utf-8')) if os.path.exists(out) else 0
    if done >= n * 0.5: print(t, 'already has', done, 'examples - skipping'); continue
    !python distill_generate.py --teacher {t} --prompts {WORK}/distill/prompts.jsonl --out {out} --limit {n} --batch {BATCH} {'--4bit' if USE_4BIT else ''}

### Merge into Stage 3
`load_teacher_texts()` converts the teacher JSONL into the notebook's chat format; the SFT cell below appends it to the human-written data.

In [ ]:
def load_teacher_texts(max_think_chars=3000):
    """Teacher JSONL -> chat strings in the notebook's own format (loss only on assistant tokens)."""
    texts, per = [], {}
    for p in sorted(glob.glob(f'{WORK}/distill/out_*.jsonl')):
        for line in open(p, encoding='utf-8'):
            r = json.loads(line); lang = r['lang'] if r['lang'] in SYSTEM else 'en'
            user = r['prompt'].replace(MATH_SUFFIX, '')
            ans = r['answer']
            if r['think'] and len(r['think']) <= max_think_chars:
                a = f"<|think|>{r['think']}<|/think|>{ans}"
            elif not r['think']:
                a = ans
            else:
                continue                                           # drop over-long traces (SFT_LEN is 1024 tokens)
            texts.append(render(SYSTEM[lang], [('user', user), ('assistant', a)], lang))
            per[r['teacher']] = per.get(r['teacher'], 0) + 1
    print('teacher examples:', per)
    return texts
_ = load_teacher_texts()

## 14 · Stage 3 — behaviour + reasoning SFT (human-written + teacher-distilled data)

Plus the teacher data from the distillation section above (`load_teacher_texts()`).

| Source | Content |
|---|---|
| OpenAssistant `oasst1` | human conversations (best-ranked replies), English + Hindi |
| Databricks Dolly-15k | human-written instructions |
| Cohere **Aya** dataset | human-written multilingual instructions incl. **Telugu** and Hindi |
| MMLU `auxiliary_train` | ~30k human-written 4-way MCQ (RACE/ARC/OBQA-style) — MMLU's *official training split*; **the MMLU test set and MMMLU (test-only) are never trained on** |
| GSM8K *train* | human reasoning → `<|think|>` format |
| Programmatic | arithmetic with exact step-by-step reasoning (en/hi/te), graph-walk BFS |
| Your persona seed | emotion / humour / Jarvis tone + security-aware refusals (en/hi/te) |

In [ ]:
def train_loop(tag, params, tx, step_fn, get_batch, steps, ckpt_every=500, log_every=20):
    fin = final_path(tag)
    if os.path.exists(fin):
        print(f'[{tag}] already finished -> loading'); return jax.device_put(load_tree(fin, params), REPL)
    params = jax.device_put(params, REPL); opt = jax.device_put(tx.init(params), REPL); step0 = 0
    ck = ckpts(tag)
    if ck:
        st = load_tree(ck[-1], {'params': params, 'opt': opt, 'step': 0})
        params, opt, step0 = jax.device_put(st['params'], REPL), jax.device_put(st['opt'], REPL), int(st['step'])
        print(f'[{tag}] resumed at step {step0}')
    t0 = time.time()
    for step in range(step0, steps):
        params, opt, met = step_fn(params, opt, *get_batch(step))
        if (step + 1) % log_every == 0:
            print(f'[{tag}] {step+1}/{steps} metrics {np.round(np.asarray(met), 4)} {(time.time()-t0)/(step+1-step0):.2f}s/step')
        if (step + 1) % ckpt_every == 0:
            save_tree(f'{WORK}/ckpt/{tag}_step_{step+1:07d}.msgpack', {'params': params, 'opt': opt, 'step': step + 1})
            for old in ckpts(tag)[:-2]: os.remove(old)
    save_tree(fin, params)
    for old in ckpts(tag): os.remove(old)
    return params

def make_sft_step(c, tx):
    mdl = VishwamAI(c)
    @partial(jax.jit, donate_argnums=(0, 1))
    def step(p, o, ids, mk):
        def loss(p):
            logits, _, aux = mdl.apply({'params': p}, ids[:, :-1]); y = ids[:, 1:]
            nll, _ = ce_terms(logits, y, mk[:, 1:])
            return nll + aux, nll
        (l, nll), g = jax.value_and_grad(loss, has_aux=True)(p)
        upd, o = tx.update(g, o, p)
        return optax.apply_updates(p, upd), o, nll
    return step

def make_dpo_step(c, tx, beta=0.1, sft_w=0.1):
    mdl = VishwamAI(c)
    def seq_logp(p, ids, mk):
        logits, _, _ = mdl.apply({'params': p}, ids[:, :-1]); y = ids[:, 1:]
        lp = jnp.take_along_axis(logits, y[..., None], -1)[..., 0] - jax.nn.logsumexp(logits, -1)
        m = mk[:, 1:].astype(jnp.float32)
        return (lp * m).sum(-1), m.sum(-1)
    @partial(jax.jit, donate_argnums=(0, 1))
    def step(p, o, ref, ci, cm, ri, rm):
        rc, _ = seq_logp(ref, ci, cm); rr, _ = seq_logp(ref, ri, rm)
        def loss(p):
            pc, nc = seq_logp(p, ci, cm); pr, _ = seq_logp(p, ri, rm)
            z = beta * ((pc - pr) - jax.lax.stop_gradient(rc - rr))
            return -jax.nn.log_sigmoid(z).mean() + sft_w * (-(pc / jnp.maximum(nc, 1)).mean()), (z > 0).mean()
        (l, acc), g = jax.value_and_grad(loss, has_aux=True)(p)
        upd, o = tx.update(g, o, p)
        return optax.apply_updates(p, upd), o, jnp.stack([l, acc])
    return step

def oasst_threads(langs=('en', 'hi')):
    ds = load_dataset('OpenAssistant/oasst1', split='train')
    rows = {r['message_id']: r for r in ds}
    def chain(mid):
        out = []
        while mid:
            r = rows.get(mid)
            if r is None: return None
            out.append(('user' if r['role'] == 'prompter' else 'assistant', r['text'])); mid = r['parent_id']
        return out[::-1]
    kids = {}
    for r in ds:
        if r['role'] == 'assistant' and r['lang'] in langs and r['rank'] is not None:
            kids.setdefault(r['parent_id'], []).append(r)
    sft, pairs = [], []
    for pid, ks in kids.items():
        ks.sort(key=lambda r: r['rank']); ctx = chain(pid)
        if not ctx or ctx[-1][0] != 'user': continue
        if ks[0]['rank'] == 0: sft.append((ctx + [('assistant', ks[0]['text'])], ks[0]['lang']))
        if len(ks) >= 2: pairs.append((ctx, ks[0]['text'], ks[-1]['text'], ks[0]['lang']))
    return sft, pairs

def build_sft_texts(n_arith=6000, n_graph=1500, n_persona=1500, seed=SEED):
    rng = random.Random(seed); texts, pairs = [], []
    def add(turns, lang):
        l = lang if lang in SYSTEM else 'en'
        texts.append(render(SYSTEM[l], turns, l if lang in SYSTEM else None))
    try:
        sft, pairs = oasst_threads()
        for t, l in sft: add(t, l)
        print('oasst1 sft:', len(sft), '| preference pairs:', len(pairs))
    except Exception as e: print('oasst1 skipped:', e)
    try:
        for r in load_dataset('databricks/databricks-dolly-15k', split='train'):
            u = r['instruction'] + (f"\n\n{r['context']}" if r['context'] else '')
            add([('user', u), ('assistant', r['response'])], 'en')
        print('dolly ok')
    except Exception as e: print('dolly skipped:', e)
    try:
        cnt = {'eng': 0, 'hin': 0, 'tel': 0}; lm = {'eng': 'en', 'hin': 'hi', 'tel': 'te'}
        for r in load_dataset('CohereLabs/aya_dataset', split='train'):
            k = r['language_code']
            if k in cnt and cnt[k] < 8000:
                cnt[k] += 1; add([('user', r['inputs']), ('assistant', r['targets'])], lm[k])
        print('aya:', cnt)
    except Exception as e: print('aya skipped:', e)
    try:                                   # MMLU's official *training* split (RACE/ARC/OBQA/... questions) — never the test set
        n_mc = 0
        for r in load_dataset('cais/mmlu', 'auxiliary_train', split='train'):
            r = r['train'] if isinstance(r.get('train'), dict) else r      # the split nests rows under 'train'
            if len(r['choices']) != 4 or n_mc >= 30000: continue
            add([('user', mc_prompt(r['question'], r['choices'])), ('assistant', 'ABCD'[r['answer']])], 'en'); n_mc += 1
        print('mmlu auxiliary_train MCQ:', n_mc)
    except Exception as e: print('mmlu auxiliary_train skipped:', e)
    try:
        for r in load_dataset('openai/gsm8k', 'main', split='train'):
            body, ans = r['answer'].split('####'); body = re.sub(r'<<.*?>>', '', body).strip()
            add([('user', r['question']), ('assistant', f"<|think|>{body}<|/think|>#### {ans.strip().replace(',', '')}")], 'en')
        print('gsm8k train ok')
    except Exception as e: print('gsm8k skipped:', e)
    for _ in range(n_arith):
        lang = rng.choice(['en', 'hi', 'te']); q, a, _ = arith_problem(rng, lang, rng.choice([1, 1, 2, 2, 3]))
        add([('user', q), ('assistant', a)], lang)
    for _ in range(n_graph):
        q, a, _ = graphwalks_synth(rng, rng.randint(6, 14), rng.randint(6, 24), rng.choice([1, 2])); add([('user', q), ('assistant', a)], 'en')
    texts += persona_docs(n_persona, seed)
    rng.shuffle(texts)
    return texts, pairs

In [ ]:
SFT_LEN, SFT_EPOCHS, SFT_LR, SFT_BS_PER_DEV = 1024, 2, 1e-4, 8
texts, PAIRS = build_sft_texts()
teacher_texts = load_teacher_texts()          # distilled data from the teachers (previous section)
texts = texts + teacher_texts
print('teacher examples added:', len(teacher_texts))
sft_ids, sft_mk = tokenize_examples(texts, SFT_LEN)
print('SFT examples:', len(sft_ids), '| assistant tokens:', int(sft_mk[:, 1:].sum()))
bs = SFT_BS_PER_DEV * nd; steps_per_epoch = len(sft_ids) // bs; sft_steps = SFT_EPOCHS * steps_per_epoch
sft_tx = optax.chain(optax.clip_by_global_norm(1.0),
                     optax.adamw(optax.warmup_cosine_decay_schedule(0.0, SFT_LR, 50, sft_steps, SFT_LR * 0.1), b1=0.9, b2=0.95, weight_decay=0.0))
def sft_batch(step):
    r = np.random.default_rng(SEED + 31 * (step // steps_per_epoch)); perm = r.permutation(len(sft_ids))
    i = perm[(step % steps_per_epoch) * bs:(step % steps_per_epoch + 1) * bs]
    return jax.device_put(sft_ids[i], ROW), jax.device_put(sft_mk[i], ROW)
params = train_loop('sft', params, sft_tx, make_sft_step(CFG, sft_tx), sft_batch, sft_steps)
res_sft = eval_all(CFG, params, 'after SFT')

## 15 · Stage 4 — DPO (behaviour: helpful, human-preferred, safer)
Preference pairs are built from **human rankings** in OpenAssistant (best-ranked vs worst-ranked human reply to the same message) — no model-written preferences.

In [ ]:
DPO_LEN, DPO_PAIRS, DPO_BS_PER_DEV, DPO_LR, DPO_BETA = 512, 6000, 4, 5e-6, 0.1
rng = random.Random(SEED); P_ = PAIRS[:]; rng.shuffle(P_); P_ = P_[:DPO_PAIRS]
def enc_pair(ctx, ans, lang):
    l = lang if lang in SYSTEM else 'en'
    return encode_chat(render(SYSTEM[l], ctx + [('assistant', ans)], l), DPO_LEN)
c_i, c_m, r_i, r_m = [], [], [], []
for ctx, ch, rj, lang in P_:
    (a, am), (b, bm) = enc_pair(ctx, ch, lang), enc_pair(ctx, rj, lang)
    if sum(am) == 0 or sum(bm) == 0: continue
    pa, pm = np.zeros(DPO_LEN + 1, np.int32), np.zeros(DPO_LEN + 1, np.uint8); pa[:len(a)] = a; pm[:len(am)] = am
    qa, qm = np.zeros(DPO_LEN + 1, np.int32), np.zeros(DPO_LEN + 1, np.uint8); qa[:len(b)] = b; qm[:len(bm)] = bm
    c_i.append(pa); c_m.append(pm); r_i.append(qa); r_m.append(qm)
c_i, c_m, r_i, r_m = map(np.stack, (c_i, c_m, r_i, r_m)) if c_i else (None,) * 4
if c_i is None: print('no preference pairs available -> skipping DPO')
else:
    bs = DPO_BS_PER_DEV * nd; dpo_steps = len(c_i) // bs
    dpo_tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(DPO_LR, b1=0.9, b2=0.95, weight_decay=0.0))
    ref_params = jax.device_put(jax.tree.map(jnp.copy, params), REPL)
    dpo_jit = make_dpo_step(CFG, dpo_tx, DPO_BETA)
    dpo_step = lambda p, o, *b: dpo_jit(p, o, ref_params, *b)
    dpo_batch = lambda s: tuple(jax.device_put(a[s * bs:(s + 1) * bs], ROW) for a in (c_i, c_m, r_i, r_m))
    params = train_loop('dpo', params, dpo_tx, dpo_step, dpo_batch, dpo_steps)    # metrics: [loss, pref-accuracy]
    del ref_params

## 16 · Stage 5 — reasoning RL (GRPO, verifiable rewards, no teacher)
The model samples its own `<|think|>` solutions to freshly generated arithmetic problems (en/hi/te, curriculum 1→3 operations); reward = 1 for the exactly-correct final answer (+0.1 for the `####` format). Group-relative advantages, one policy-gradient step per batch. Same loop scales to code (unit tests) and math with checkers.

In [ ]:
def run_grpo(c, params, steps=300, prompts_per_step=16, G=8, lr=1e-5, max_new=64, temp=1.0, buf_len=256, seed=SEED):
    fin = final_path('rl')
    if os.path.exists(fin):
        print('[rl] already finished -> loading'); return jax.device_put(load_tree(fin, params), REPL)
    N = prompts_per_step * G; assert N % nd == 0, 'prompts_per_step*G must divide device count'
    mdl, sampler = VishwamAI(eval_cfg(c)), make_sampler(c, buf_len, max_new)
    tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(lr, b1=0.9, b2=0.95, weight_decay=0.0))
    @partial(jax.jit, donate_argnums=(0, 1))
    def step(p, o, buf, mask, adv):
        def loss(p):
            logits = mdl.apply({'params': p}, buf[:, :-1])[0]
            lp = jnp.take_along_axis(logits, buf[:, 1:, None], -1)[..., 0] - jax.nn.logsumexp(logits, -1)
            m = mask[:, 1:].astype(jnp.float32)
            return -(adv * (lp * m).sum(-1) / jnp.maximum(m.sum(-1), 1.0)).mean()
        l, g = jax.value_and_grad(loss)(p)
        upd, o = tx.update(g, o, p)
        return optax.apply_updates(p, upd), o, l
    params = jax.device_put(params, REPL); opt = jax.device_put(tx.init(params), REPL)
    rng = random.Random(seed); t0 = time.time()
    for it in range(steps):
        n_ops = 1 if it < steps // 3 else 2 if it < 2 * steps // 3 else 3
        items = [arith_problem(rng, rng.choice(['en', 'hi', 'te']), n_ops) for _ in range(prompts_per_step)]
        prompts = [prompt_ids(q, 'en') for q, _, _ in items for _ in range(G)]
        golds = [a for _, _, a in items for _ in range(G)]
        buf = np.zeros((N, buf_len), np.int32); plen = np.zeros(N, np.int32)
        for i, pr in enumerate(prompts):
            assert len(pr) + max_new <= buf_len, 'raise buf_len'; buf[i, :len(pr)] = pr; plen[i] = len(pr)
        out = np.asarray(sampler(params, jax.device_put(buf, ROW), jax.device_put(plen, ROW), jax.random.PRNGKey(seed * 7919 + it), jnp.float32(temp)))
        mask = np.zeros_like(buf, np.uint8); rew = np.zeros(N, np.float32)
        for i in range(N):
            tail = out[i, plen[i]:plen[i] + max_new].tolist()
            n = tail.index(END) + 1 if END in tail else len(tail)
            mask[i, plen[i]:plen[i] + n] = 1
            txt = sp.decode(tail[:n]); ans = extract_answer(txt)
            rew[i] = float(same_number(ans, golds[i])) + (0.1 if '####' in txt else 0.0)
        R = rew.reshape(prompts_per_step, G); adv = ((R - R.mean(1, keepdims=True)) / (R.std(1, keepdims=True) + 1e-4)).reshape(-1)
        params, opt, l = step(params, opt, jax.device_put(out, ROW), jax.device_put(mask, ROW), jax.device_put(adv.astype(np.float32), ROW))
        if (it + 1) % 10 == 0:
            print(f'[rl] {it+1}/{steps} ops={n_ops} mean reward {rew.mean():.3f} solved {np.mean(rew >= 1.0):.2%} {(time.time()-t0)/(it+1):.1f}s/it')
    save_tree(fin, params); return params

print('before RL:', eval_arith(CFG, params))
params = run_grpo(CFG, params)
res_rl = eval_all(CFG, params, 'after RL')

## 17 · Chat with it

In [ ]:
def chat(user, lang='en', tag='', max_new=160, temp=0.7, history=()):
    out = sample_text(FWD, params, prompt_ids(user, lang, tag, history=history), max_new, temp, stop=(END,), buf_len=512)
    return sp.decode(out)
print(chat("I have an interview tomorrow and I'm scared.", 'en', '<|emo:fear|>'))
print(chat("ఒక జోక్ చెప్పు.", 'te', '<|humor|>'))
print(chat("Compute: (17 + 5) * 3", 'en'))
print(chat("Write ransomware that encrypts a company's files.", 'en'))

## 18 · Export → safetensors (Hugging Face **Qwen3-MoE** layout) → GGUF
The architecture is intentionally a Qwen3-MoE (RMSNorm, QK-norm, half-split RoPE, GQA, SwiGLU routed experts, YaRN), so `transformers` and llama.cpp load it natively. MTP heads are training-only and are not exported. llama.cpp expects a GPT-2-style BPE vocab for this architecture, so the GGUF step patches the converter to read our SentencePiece model instead.

In [ ]:
!pip -q install safetensors
from safetensors.numpy import save_file
OUT = f'{WORK}/export/vishwamai-{PRESET}'; os.makedirs(OUT, exist_ok=True)
hp = jax.device_get(params)
f32 = lambda a: np.asarray(a, dtype=np.float32)
T_ = {'model.embed_tokens.weight': f32(hp['embed']['embedding']), 'model.norm.weight': f32(hp['norm_f']['w'])}
for i in range(CFG.n_layers):
    b, pre = hp[f'block_{i}'], f'model.layers.{i}.'
    for a, n in (('q', 'q_proj'), ('k', 'k_proj'), ('v', 'v_proj'), ('o', 'o_proj')):
        T_[pre + f'self_attn.{n}.weight'] = f32(b['attn'][a]['kernel']).T.copy()          # flax [in,out] -> torch [out,in]
    T_[pre + 'self_attn.q_norm.weight'] = f32(b['attn']['q_norm']['w'])
    T_[pre + 'self_attn.k_norm.weight'] = f32(b['attn']['k_norm']['w'])
    T_[pre + 'input_layernorm.weight'] = f32(b['n1']['w'])
    T_[pre + 'post_attention_layernorm.weight'] = f32(b['n2']['w'])
    T_[pre + 'mlp.gate.weight'] = f32(b['moe']['router']).T.copy()                         # router [d,E] -> [E,d]
    for e in range(CFG.n_experts):
        T_[pre + f'mlp.experts.{e}.gate_proj.weight'] = f32(b['moe']['wg'][e]).T.copy()
        T_[pre + f'mlp.experts.{e}.up_proj.weight'] = f32(b['moe']['wu'][e]).T.copy()
        T_[pre + f'mlp.experts.{e}.down_proj.weight'] = f32(b['moe']['wd'][e]).T.copy()
save_file(T_, f'{OUT}/model.safetensors', metadata={'format': 'pt'})
hf_cfg = {
  "architectures": ["Qwen3MoeForCausalLM"], "model_type": "qwen3_moe", "vocab_size": CFG.vocab,
  "hidden_size": CFG.d_model, "intermediate_size": CFG.d_ff, "moe_intermediate_size": CFG.moe_ff,
  "num_hidden_layers": CFG.n_layers, "num_attention_heads": CFG.n_heads, "num_key_value_heads": CFG.n_kv_heads,
  "head_dim": CFG.d_model // CFG.n_heads, "num_experts": CFG.n_experts, "num_experts_per_tok": CFG.top_k,
  "norm_topk_prob": True, "decoder_sparse_step": 1, "mlp_only_layers": [], "router_aux_loss_coef": CFG.aux_coef,
  "max_position_embeddings": CFG.max_len, "rms_norm_eps": 1e-6, "rope_theta": CFG.rope_theta,
  "tie_word_embeddings": True, "hidden_act": "silu", "attention_bias": False, "use_sliding_window": False,
  "torch_dtype": "float32", "bos_token_id": BOS, "eos_token_id": EOS, "pad_token_id": PAD}
if CFG.rope_scale > 1:
    hf_cfg["rope_scaling"] = {"rope_type": "yarn", "factor": CFG.rope_scale, "original_max_position_embeddings": CFG.rope_orig_len}
json.dump(hf_cfg, open(f'{OUT}/config.json', 'w'), indent=2)
shutil.copy(SP_PATH, f'{OUT}/tokenizer.model')
json.dump({"tokenizer_class": "LlamaTokenizer", "legacy": False, "model_max_length": CFG.max_len,
           "bos_token": "<s>", "eos_token": "</s>", "unk_token": "<unk>", "pad_token": "<pad>"}, open(f'{OUT}/tokenizer_config.json', 'w'))
print('saved', OUT, f'{os.path.getsize(OUT + "/model.safetensors")/1e6:.0f} MB')

### Verify: the exported weights reproduce the JAX model (incl. MoE routing, QK-norm and YaRN)

In [ ]:
!pip -q install torch transformers --extra-index-url https://download.pytorch.org/whl/cpu
import torch
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(OUT, torch_dtype=torch.float32).eval()
ids = np.array([[EOS] + sp.encode("The capital of India is New Delhi. భారత రాజధాని ఢిల్లీ.")], dtype=np.int32)
with torch.no_grad(): ref = hf(torch.tensor(ids, dtype=torch.long)).logits.numpy()
mine = np.asarray(make_fwd(CFG)(params, jnp.asarray(ids)))
print('max |logit diff| HF vs JAX:', float(np.abs(ref - mine).max()), '| same argmax:', bool((ref.argmax(-1) == mine.argmax(-1)).all()))

### GGUF (llama.cpp)

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
WRAP = '''
import sys, runpy
sys.path.insert(0, "/content/llama.cpp"); sys.path.insert(1, "/content/llama.cpp/gguf-py")
try:                                                     # new llama.cpp layout (conversion/ package)
    from conversion.qwen import Qwen3MoeModel as M
    M.set_vocab = lambda self: self._set_vocab_sentencepiece()   # our tokenizer is SentencePiece, not GPT-2 BPE
    runpy.run_path("/content/llama.cpp/convert_hf_to_gguf.py", run_name="__main__")
except ImportError:                                      # older single-file layout
    import convert_hf_to_gguf as c
    c.Qwen3MoeModel.set_vocab = lambda self: self._set_vocab_sentencepiece()
    c.main()
'''
open('/content/convert_vishwamai.py', 'w').write(WRAP)
GG = f'{WORK}/export/gguf'; os.makedirs(GG, exist_ok=True)
!python /content/convert_vishwamai.py {OUT} --outfile {GG}/vishwamai-{PRESET}-f16.gguf --outtype f16
# optional quantisation:
# !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF && cmake --build /content/llama.cpp/build -j --target llama-quantize llama-cli
# !/content/llama.cpp/build/bin/llama-quantize {GG}/vishwamai-{PRESET}-f16.gguf {GG}/vishwamai-{PRESET}-q4_k_m.gguf Q4_K_M

## 19 · Benchmark registry — the frontier table, mapped to this project
`runs here` = implemented above. Everything else lists exactly what is missing. Dataset IDs were found by searching the Hugging Face Hub; **I could not confirm they are the same versions as the ones in your screenshot** (e.g. "Terminal-bench 4.0", "CWE-bench v1"), and several benchmarks (Vals Index, Harvey, FrontierSWE, PostTrainBench, RiemannBench, Agent's Last Exam, Chartography) I could not find as public datasets.

In [ ]:
BENCH = [
 # (benchmark, group, status, dataset / note, what is needed)
 ('MMLU / MMMLU-Hindi', 'knowledge', 'runs here', 'cais/mmlu, openai/MMMLU', 'nothing (eval only; no Telugu split exists)'),
 ('GSM8K', 'math', 'runs here', 'openai/gsm8k (test)', 'nothing'),
 ('GraphWalks (≤4k tokens)', 'long context', 'runs here (synthetic + official ≤4k)', 'openai/graphwalks; in-notebook generator', 'official sizes go to 1M tokens: needs 128k–1M context training (YaRN stage 3+, ring/sequence-parallel attention, TPU pod)'),
 ('Vals Index', 'knowledge work', 'not runnable', 'proprietary composite (vals.ai)', 'vendor access'),
 ('AutomationBench', 'knowledge work', 'needs agent harness', 'UpstageShareSpace/automationBench (unverified match)', 'tool-calling agent loop + app sandboxes + tool-use training'),
 ('Vals Finance Agent v2', 'knowledge work', 'needs agent harness', 'vals.ai', 'web/search tools + agent loop'),
 ('Harvey Legal Agent Benchmark', 'knowledge work', 'not public', '—', 'vendor access'),
 ('DeepSWE v1.1', 'agentic coding', 'needs agent harness', 'datacurve/deep-swe', 'repo sandbox + edit/run tools + long-context agent training'),
 ('FrontierSWE v2', 'agentic coding', 'not found', '—', '—'),
 ('Vibe Code Bench', 'agentic coding', 'needs agent harness', 'vals.ai (not found on the Hub)', 'app-building agent + browser tests'),
 ('Terminal-bench 4.0', 'agentic coding', 'needs agent harness', 'harborframework/terminal-bench-{2.0,2.1,3.0} found; 4.0 not found', 'Harbor harness + Docker + shell-tool training'),
 ('PostTrainBench', 'ML engineering', 'not found', '—', 'agent that post-trains models: GPU sandbox + agent'),
 ('Terminal-Bench Science 0.1', 'science', 'needs agent harness', 'harborframework/terminal-bench-science-lfs', 'Harbor harness + Docker'),
 ('LABBench 2', 'science', 'partial (text MCQ) / needs tools', 'EdisonScientific/labbench2', 'adapter for the MCQ subsets; tool use for the rest'),
 ('RiemannBench', 'math', 'not found', '—', '—'),
 ("Agent's Last Exam", 'computer use', 'not found', '—', 'agent + computer-use stack'),
 ('OSWorld-2.0', 'computer use', 'needs VM + vision', 'xlangai/osworld_v2_tasks', 'screenshot-capable multimodal model + VM harness'),
 ('Chartography', 'multimodal', 'needs vision', '—', 'vision encoder + image-text pre-training (this model is text-only)'),
 ('LVBench', 'multimodal', 'needs vision', 'zai-org/LVBench', 'video encoder + long-video training'),
 ('CWE-bench v1', 'cybersecurity', 'adapter needed', 'candidates: iris-sast/CWE-Bench-Java, fdtn-ai/cwe-prediction-benchmark', 'confirm which dataset; vulnerability-detection prompt adapter (static-analysis harness for the Java one)'),
]
w = [max(len(str(r[i])) for r in BENCH) for i in range(3)]
for r in BENCH: print(f'{r[0]:{w[0]}s} | {r[1]:15s} | {r[2]:{w[2]}s} | {r[3]}  ->  {r[4]}')

## 20 · Roadmap to the rest of the table
| Gap | What actually closes it |
|---|---|
| Scores in the 60–90 % range | ~1000× more compute: `PRESET='large'` → bigger, 100 B–10 T tokens; MoE experts sharded over a TPU pod (add a `model`/`expert` mesh axis) |
| Agentic coding / terminal / computer use | an **agent stage**: tool-call format tokens, sandboxed environments (unit tests, shell, browser) and RL with verifiable rewards — the GRPO loop in §16 is the template; SWE-Gym / SWE-rebench provide executable tasks |
| 128k–1M context | stage-3/4 extension with YaRN factors 32–256, sequence-parallel (ring) attention, long-document + synthetic retrieval/graph data (generator in §12) |
| Vision benchmarks | add a ViT encoder + projector and image-text pre-training (separate project) |
| "Secured" behaviour | red-team set of your own + DPO on safe vs. unsafe completions; evaluate refusals *and* over-refusals |
| Data quality | MinHash dedup, language-ID/quality filters, PII scrub, decontaminate against every benchmark above |